# Path train data generation

This notebook generates the paths that will be used for the path classification model training. It uses the segmentation predictions generated in the [previous notebook (3)](./03_generate_segmentation_preds.ipynb) to create the paths.

Different path generation methods can be used, but the one used in this notebook is the one detailed in the original paper, which consists in using the segmentation prediction and the ground truth to find the parts of vessels that have been forgotten by the segmentation model. The paths are then generated by connecting the endpoints of these parts to the rest of the vessel tree, and are constituted of a list of coordinates on the image, that follow the an euclidean shortest path between the endpoints.

In [1]:
from utils.available_datasets import available_datasets

dataset_choice = available_datasets["PERSEVERE"]

train_split = dataset_choice.preferred_train_split
data_dir = dataset_choice.data_dir
splits_filepath = dataset_choice.splits_filepath
ndim = dataset_choice.ndim
input_channels = dataset_choice.input_channels

In [2]:
import os
from image_segmentation.data import ImageDatamodule
from image_segmentation.data.image_dataset import FundusImageDataset

datamodule = ImageDatamodule(
    data_dir=data_dir,
    split_file_path=splits_filepath,
    train_split_name=train_split,
)
datamodule.setup()
dataset = datamodule.dataset

stats = dataset.get_dataset_stats(ndim=ndim, input_channels=input_channels, split_name=train_split, split_indices=datamodule.train_indices)

distances_hparams = None
if isinstance(dataset, FundusImageDataset):
    dataset_res = stats["estimated_resolution_mm_per_pixel"]
    distances_hparams = dataset.get_dataset_distance_hparams()

Dataset split: Train=226, Val=56, Test=71
Loading dataset stats for split 'train' from /home/morand/afs/EVAPORE/data/PERSEVERE/image_stats.json...


In [3]:
gt_folder = os.path.join(data_dir, "gt")

main_centerlines_folder = os.path.join(data_dir, "centerlines")
os.makedirs(main_centerlines_folder, exist_ok=True)

As stated in the paper, we only consider centerlines with an euclidean length less than 100 pixels, as longer centerlines are more likely to drift away from the euclidean path bewteen endpoints. Becasue of that, the feature sampling is more likely to be irrelevant and thus those paths are more likely to be wrongly labeled by the model.

If you want to use all the centerlines, just set max_dist to None.

In [4]:
from math import ceil

max_dist = 100
if distances_hparams is not None:
    max_dist = int(ceil(distances_hparams["max_dist"]))
    max_dist_mm = FundusImageDataset.length_in_pixels_to_mm(max_dist, dataset_res)
    print(f"Generating centerlines with max distance of {max_dist} pixels ({max_dist_mm:.2f} mm)")

if max_dist is None:
    centerlines_folder = os.path.join(main_centerlines_folder, f"euclidean_all_centerlines")
else:
    centerlines_folder = os.path.join(main_centerlines_folder, f"euclidean_lt_{max_dist}_centerlines")
os.makedirs(centerlines_folder, exist_ok=True)

In [5]:
oversampling_max_dist = 20
if distances_hparams is not None:
    oversampling_max_dist = distances_hparams["oversampling_size"]
    oversampling_max_dist_mm = FundusImageDataset.length_in_pixels_to_mm(oversampling_max_dist, dataset_res)
    print(f"Oversampling nodes with max distance of {oversampling_max_dist} pixels ({oversampling_max_dist_mm:.2f} mm)")

n_closest = 3

# Construct the path dataset (positives and negatives samples)

In [6]:
from graph.graph_visualization import display_graph_overlay

In [7]:
import networkx as nx
import torch
import numpy as np
from skimage.morphology import dilation, disk
from skimage.measure import label
from typing import List


def get_query_edges(graph: nx.Graph, n_closest: int = 1, max_dist: float = None) -> torch.Tensor:
    G = graph.copy()

    connected_components = list(nx.connected_components(graph))

    cc_extrimities = {}
    for cc_i, cc in enumerate(connected_components):
        extrimities = []
        for n in cc:
            if G.degree(n) == 1:
                extrimities.append(n)
        cc_extrimities[cc_i] = extrimities

    cc_n_count = [len(cc) for cc in connected_components]
    main_cc = np.argmax(cc_n_count)
    small_ccs = {i: cc for i, cc in enumerate(connected_components) if i != main_cc}

    virtual_edges_to_add = []
    for i, cc in small_ccs.items():
        extremities = cc_extrimities[i]
        other_ccs = {j: other_cc for j, other_cc in enumerate(connected_components) if j != i}
        other_ccs_all_nodes = []
        for other_cc in other_ccs.values():
            other_ccs_all_nodes.extend(other_cc)

        for extrimity in extremities:
            other_nodes_distances = []
            extrimity_pos = np.array(G.nodes[extrimity]['pos'])
            for other_cc_node in other_ccs_all_nodes:
                other_cc_node_pos = np.array(G.nodes[other_cc_node]['pos'])
                dist = np.linalg.norm(extrimity_pos - other_cc_node_pos)
                other_nodes_distances.append(dist)
            other_nodes_distances = np.array(other_nodes_distances)
            closest_indices = np.argsort(other_nodes_distances)[:n_closest]
            if max_dist is not None:
                closest_indices = closest_indices[other_nodes_distances[closest_indices] <= max_dist]
            closest_nodes = [other_ccs_all_nodes[idx] for idx in closest_indices]

            for extremity_closest_other_cc_node in closest_nodes:
                if (extrimity, extremity_closest_other_cc_node) not in virtual_edges_to_add and (extremity_closest_other_cc_node, extrimity) not in virtual_edges_to_add:
                    virtual_edges_to_add.append([extrimity, extremity_closest_other_cc_node])
    
    virtual_edges_index_tensor = torch.tensor(virtual_edges_to_add, dtype=torch.long).t().contiguous()
    return virtual_edges_index_tensor


# ======== Cleaning paths functions ========

def cut_mask_from_negative_edges_for_all(centerlines, mask, classes=None, edges=None, return_old_centerlines=False):
    not_mask = np.logical_not(mask)

    new_centerlines, new_classes, new_edges, old_centerlines = [], [], [], []

    for i, centerline in enumerate(centerlines):
        keep = not_mask[tuple(np.asarray(centerline).T)]

        runs, current = [], []
        for coords, k in zip(centerline, keep):
            if k:
                current.append([int(c) for c in coords])
            else:
                if current:
                    runs.append(current)
                    current = []
        if current:
            runs.append(current)

        n_new = len(runs)
        if n_new > 0:
            new_centerlines.extend(runs)
            if return_old_centerlines:
                old_centerlines.extend([centerline] * n_new)
            if classes is not None:
                new_classes.extend([classes[i]] * n_new)
            if edges is not None:
                new_edges.extend([edges[i]] * n_new)

    res = {"new_centerlines": new_centerlines}
    if return_old_centerlines:
        res["old_centerlines"] = old_centerlines
    if classes is not None:
        res["classes"] = new_classes
    if edges is not None:
        res["edges"] = new_edges
    return res


def is_reconstructed_path_not_too_far(
    true_path_existing_centerline: np.ndarray,      # (M, 2) or (M, 3)
    true_path_reconstructed_centerline: np.ndarray,  # (N, 2) or (N, 3)
    distance_ratio_threshold: float
) -> bool:
    existing = np.asarray(true_path_existing_centerline, dtype=float)
    reconstructed = np.asarray(true_path_reconstructed_centerline, dtype=float)

    # pairwise distances: (N, M)
    diffs = reconstructed[:, None, :] - existing[None, :, :]
    dists = np.linalg.norm(diffs, axis=-1)
    min_dists = dists.min(axis=1)

    sum_min_distances = min_dists.sum() / len(reconstructed)
    return sum_min_distances <= distance_ratio_threshold


def remove_too_far_reconstructed_paths_for_all(
    true_path_existing_centerlines: List[np.ndarray],
    true_path_reconstructed_centerlines: List[np.ndarray],
    distance_ratio_threshold: float
) -> List[int]:
    new_reconstructed_classes = []
    for i, true_path_reconstructed_centerline in enumerate(true_path_reconstructed_centerlines):
        true_path_existing_centerline = true_path_existing_centerlines[i]
        condition = is_reconstructed_path_not_too_far(
            true_path_existing_centerline,
            true_path_reconstructed_centerline,
            distance_ratio_threshold
        )
        new_reconstructed_classes.append(int(condition))
    return new_reconstructed_classes

from skimage.morphology import dilation, disk, ball

def clean_paths_on_surface_of_mask(centerlines, mask, kernel_size=1, threshold=0.5, classes=None):
    new_centerlines = []
    if classes is not None:
        new_classes = []

    labeled_mask = label(mask)
    footprint = disk(kernel_size) if mask.ndim == 2 else ball(kernel_size)
    dilated_mask = dilation(labeled_mask, footprint)  # computed once, unchanged

    for i, centerline in enumerate(centerlines):
        coords_arr = np.asarray(centerline)
        # Direct index instead of building a full-volume mask + elementwise multiply
        values_on_points = dilated_mask[tuple(coords_arr.T)]

        nonzero_values = values_on_points[values_on_points != 0]
        values_on_mask = len(nonzero_values)
        n_diff_cc = len(np.unique(nonzero_values))
        ratio_on_mask = values_on_mask / len(centerline)

        if ratio_on_mask <= threshold or n_diff_cc > 1:
            new_centerlines.append(centerline)
            if classes is not None:
                new_classes.append(classes[i])

    if classes is not None:
        return new_centerlines, new_classes
    return new_centerlines


In [8]:
import json
import matplotlib.pyplot as plt
from PIL import Image
import numpy as np
import networkx as nx
import torch
import os
from typing import List, Tuple
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from skimage import measure

from graph.graph_pred_state import get_combined_graph, get_combined_graph_optim
from graph.graph_pred_state import EdgePredState
from graph.graph_oversampling import OversampleNodesTransform
from graph.graph_wrapper import GraphWrapper
from utils.reconstruction.path_reconstruction.euclidean_path_reconstruction import EuclideanPathReconstructionMethod
from image_segmentation.data.io_utils import load_array

path_reconstruction_method = EuclideanPathReconstructionMethod()

def get_positive_samples(
    new_nx_graph: nx.Graph,
    pred_np: np.ndarray,
    max_dist: float
) -> Tuple[List[List[List[int]]], List[int]]:
    '''
    Computes positive samples for training the path reconstruction model.
    It identifies edges that are in the ground truth but not in the prediction and get the paths for these edges.
    The reconstructed paths are then filtered to get clean positive samples for training the model.

    Args:
        new_nx_graph (nx.Graph): The graph containing all the edges (both in prediction and not in prediction).
        pred_np (np.ndarray): The numpy array representing the prediction mask.
        max_dist (float): The maximum distance threshold to consider for true path edges.

    Returns:
        tuple[list, list]: A tuple containing the list of positive centerlines and their corresponding classes (all 1)
    '''
    true_path_edges = []
    for u, v, d in new_nx_graph.edges(data=True):
        edge_pred_state = d.get("edge_pred_state", None)
        u_pos, v_pos = new_nx_graph.nodes[u]['pos'], new_nx_graph.nodes[v]['pos']
        if edge_pred_state in [EdgePredState.NOT_IN_PREDICTION, EdgePredState.NOT_IN_PREDICTION.value]:
            euclidean_dist = np.linalg.norm(np.array(u_pos) - np.array(v_pos))
            if u_pos != v_pos and euclidean_dist <= max_dist:
                true_path_edges.append([u, v])

    true_path_edges = torch.tensor(true_path_edges, dtype=torch.long).t().contiguous()
    train_positive_centerlines = path_reconstruction_method.reconstruct(map=None, graph=new_nx_graph, new_edges=true_path_edges)
    train_positive_centerlines = [[[int(c) for c in coords] for coords in path] for path in train_positive_centerlines]
    train_positive_centerlines = cut_mask_from_negative_edges_for_all(train_positive_centerlines, pred_np)["new_centerlines"]
    train_positive_centerlines = clean_paths_on_surface_of_mask(train_positive_centerlines, pred_np, kernel_size=2, threshold=0.5)
    train_positive_classes = [1] * len(train_positive_centerlines)

    return train_positive_centerlines, train_positive_classes

def get_negative_samples(
    new_nx_graph: nx.Graph,
    in_pred_graph: nx.Graph,
    pred_np: np.ndarray,
    gt_np: np.ndarray,
    max_dist: float,
    n_closest: int,
) -> Tuple[List[List[List[int]]], List[int]]:
    '''
    Computes negative samples for training the path reconstruction model.
    It identifies edges that are not present in the graph and get the paths for these edges.
    The reconstructed paths are then filtered to get clean negative samples for training the model.

    Args:
        new_nx_graph (nx.Graph): The graph containing all the edges (both in prediction and not in prediction).
        in_pred_graph (nx.Graph): The graph containing only the edges that are in the prediction.
        pred_np (np.ndarray): The numpy array representing the prediction mask.
        gt_np (np.ndarray): The numpy array representing the ground truth mask.
        max_dist (float): The maximum distance threshold to consider for negative path edges.
        n_closest (int): The number of closest nodes to consider when generating negative samples.

    Returns:
        tuple[list, list]: A tuple containing the list of negative centerlines and their corresponding classes (all 0)
    '''
    false_query_edges = get_query_edges(in_pred_graph, n_closest=n_closest, max_dist=max_dist).t().contiguous().tolist()
    train_negative_edges = []
    for (u, v) in false_query_edges:
        if not new_nx_graph.has_edge(u, v) and not new_nx_graph.has_edge(v, u):
            train_negative_edges.append([u, v])
    train_negative_edges = torch.tensor(train_negative_edges, dtype=torch.long).t().contiguous()

    train_negative_centerlines = path_reconstruction_method.reconstruct(map=None, graph=new_nx_graph, new_edges=train_negative_edges)
    train_negative_centerlines = [[[int(c) for c in coords] for coords in path] for path in train_negative_centerlines]
    train_negative_centerlines = cut_mask_from_negative_edges_for_all(train_negative_centerlines, pred_np)["new_centerlines"]
    train_negative_centerlines = clean_paths_on_surface_of_mask(train_negative_centerlines, pred_np, kernel_size=2, threshold=0.5)
    train_negative_centerlines = clean_paths_on_surface_of_mask(train_negative_centerlines, gt_np, kernel_size=0, threshold=0.5)
    train_negative_classes = [0] * len(train_negative_centerlines)

    return train_negative_centerlines, train_negative_classes

def display_case_2d(
    img: np.ndarray,
    gt_np: np.ndarray,
    pred_np: np.ndarray,
    true_path_centerlines: List[List[List[int]]],
    train_negative_centerlines: List[List[List[int]]],
) -> None:
    '''
    2D-only visualization. For 3D volumes, this needs a different approach
    (e.g. slice-by-slice display, or a max-intensity projection) — plt.imshow
    cannot render a 3D array directly.
    '''
    mask_img = np.zeros((*gt_np.shape, 3), dtype=np.uint8)
    mask_img[gt_np] = np.array([255, 0, 0], dtype=np.uint8)  # Red for false negatives
    mask_img[pred_np] += np.array([0, 255, 255], dtype=np.uint8)  # Cyan for false positives

    centerlines_img = np.zeros((*gt_np.shape, 3), dtype=np.uint8)
    centerlines_img[pred_np] = [255, 255, 255]
    for centerline in true_path_centerlines:
        for coords in centerline:
            centerlines_img[tuple(coords)] = [0, 255, 0]  # Green for true path centerlines
    for centerline in train_negative_centerlines:
        for coords in centerline:
            centerlines_img[tuple(coords)] = [255, 0, 0]  # Red for negative samples

    fig, axs = plt.subplots(1, 3, figsize=(40, 20))
    axs[0].imshow(img)
    axs[0].set_title("Original Image")
    axs[0].axis('off')
    axs[1].imshow(mask_img)
    axs[1].set_title("Prediction and GT: TP (white), FN (red), FP (cyan)")
    axs[1].axis('off')
    axs[2].imshow(centerlines_img)
    axs[2].set_title("Prediction mask and centerlines: positives samples (green), negative samples (red)")
    axs[2].axis('off')
    plt.show()


def display_case_3d(
    pred_np: np.ndarray,
    true_path_centerlines: List[List[List[int]]],
    train_negative_centerlines: List[List[List[int]]],
    graph: nx.Graph,
) -> None:
    """
    3D visualization equivalent of display_case_2d, showing only the
    prediction volume plus centerlines:
      - green = true path centerlines
      - red   = negative sample centerlines
      - graph edge centerlines colored by edge_pred_state
        (green = IN_PREDICTION, red = NOT_IN_PREDICTION, blue = other)
    """
    pred_np = pred_np.astype(bool)

    # extract surface of the prediction volume
    verts, faces, _, _ = measure.marching_cubes(pred_np.astype(np.uint8), level=0.5)

    fig = go.Figure(
        data=[
            go.Mesh3d(
                x=verts[:, 0],
                y=verts[:, 1],
                z=verts[:, 2],
                i=faces[:, 0],
                j=faces[:, 1],
                k=faces[:, 2],
                color="lightgray",
                opacity=0.3,
            )
        ]
    )

    centerline_indices = []

    for centerline in true_path_centerlines:
        coords = np.array(centerline)
        if coords.size == 0:
            continue
        fig.add_trace(
            go.Scatter3d(
                x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
                mode="lines",
                line=dict(color="blue", width=5),
                showlegend=False,
            )
        )
        centerline_indices.append(len(fig.data) - 1)

    for centerline in train_negative_centerlines:
        coords = np.array(centerline)
        if coords.size == 0:
            continue
        fig.add_trace(
            go.Scatter3d(
                x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
                mode="lines",
                line=dict(color="yellow", width=5),
                showlegend=False,
            )
        )
        centerline_indices.append(len(fig.data) - 1)

    # centerlines from the graph edges
    graph_centerline_indices = []
    for u, v, data in graph.edges(data=True):
        centerline = data.get("centerline")
        if centerline is None:
            continue
        coords = np.array(centerline)
        if coords.size == 0:
            continue

        edge_pred_state = data.get("edge_pred_state", None)
        color = "red"
        if edge_pred_state is not None:
            if edge_pred_state.value == EdgePredState.IN_PREDICTION.value:
                color = "green"
            elif edge_pred_state.value == EdgePredState.NOT_IN_PREDICTION.value:
                color = "red"
            else:
                color = "blue"

        fig.add_trace(
            go.Scatter3d(
                x=coords[:, 0], y=coords[:, 1], z=coords[:, 2],
                mode="lines",
                line=dict(color=color, width=5),
                showlegend=False,
            )
        )
        graph_centerline_indices.append(len(fig.data) - 1)

    n_mesh = 1
    n_manual = len(centerline_indices)
    n_graph = len(graph_centerline_indices)

    fig.update_layout(
        showlegend=False,
        width=1200,
        height=800,
        scene=dict(aspectmode="data"),
        updatemenus=[
            dict(
                type="buttons",
                buttons=[
                    dict(
                        label="Hide mesh",
                        method="update",
                        args=[{"visible": [False] + [True] * (n_manual + n_graph)}],
                    ),
                    dict(
                        label="Hide manual centerlines",
                        method="update",
                        args=[{"visible": [True] + [False] * n_manual + [True] * n_graph}],
                    ),
                    dict(
                        label="Hide graph centerlines",
                        method="update",
                        args=[{"visible": [True] + [True] * n_manual + [False] * n_graph}],
                    ),
                    dict(
                        label="Show all",
                        method="update",
                        args=[{"visible": [True] * (n_mesh + n_manual + n_graph)}],
                    ),
                ],
            )
        ],
    )

    fig.show()

def process_case(i: int, 
                 data_dir: str,
                 centerlines_folder: str,
                 centerline_max_dist: float,
                 oversampling_max_dist: float,
                 n_closest: int,
                 display: bool = False,
                 debug: bool = False) -> None:
    '''
    Process a single case to generate training data for path reconstruction, and save the generated centerlines and their classes to a JSON file.

    Args:
        i (int): Index of the case to process.
        centerline_max_dist (float): The maximum distance threshold to consider for path when generating samples.
        oversampling_max_dist (float): The maximum distance threshold to consider when oversampling nodes in the graph.
        n_closest (int): The number of closest nodes to consider when generating negative samples.
        display (bool): Whether to display the case being processed (default: False).
    '''

    gt_folder = os.path.join(data_dir, "gt")
    pred_folder = os.path.join(data_dir, "pred")
    img_folder = os.path.join(data_dir, "img")

    gt_path_list = os.listdir(gt_folder)
    gt_path_list.sort()

    filename = gt_path_list[i]
    base_name = os.path.basename(filename)
    gt_path = os.path.join(gt_folder, filename)
    pred_path = os.path.join(pred_folder, filename)
    centerline_path = os.path.join(centerlines_folder, (base_name.split(".")[0] + ".json"))

    if not debug and os.path.exists(centerline_path):
        print(f"Centerline file already exists for {filename}, skipping...")
        return

    gt_np = load_array(gt_path, grayscale=True) > 0
    pred_np = load_array(pred_path, grayscale=True) > 0

    if gt_np.sum() == 0 or pred_np.sum() == 0:
        train_data = {"path_centerlines": [], "edges_classes": []}
        with open(centerline_path, 'w') as f:
            json.dump(train_data, f)
        print(f"No vessels in GT or prediction for {filename}, skipping...")
        return

    # Train data creation
    combined_graph: nx.Graph = get_combined_graph_optim(gt_np, pred_np)
    #display_graph_overlay(gt_np, combined_graph)
    #display_graph_overlay(pred_np, combined_graph)
    
    oversample_nodes_transform = OversampleNodesTransform(oversampling_max_dist, remove_original_edges=True)
    graph_wrapper: GraphWrapper = oversample_nodes_transform(GraphWrapper(combined_graph))
    new_nx_graph: nx.Graph = graph_wrapper.get_graph()
    in_pred_graph: nx.Graph = graph_wrapper.in_pred_graph
    #display_graph_overlay(pred_np, new_nx_graph)

    # Get positive and negative samples
    true_path_centerlines, true_edges_classes = get_positive_samples(new_nx_graph, pred_np, centerline_max_dist)
    train_negative_centerlines, train_negative_classes = get_negative_samples(new_nx_graph, in_pred_graph, pred_np, gt_np, centerline_max_dist, n_closest)

    if display:
        if gt_np.ndim != 2:
            display_case_3d(pred_np, true_path_centerlines, train_negative_centerlines, new_nx_graph)
        else:
            img = np.array(Image.open(os.path.join(img_folder, filename)).convert("RGB"))
            display_case_2d(img, gt_np, pred_np, true_path_centerlines, train_negative_centerlines)

    # Combine positive and negative samples to create the training data
    train_data =  {
        "path_centerlines": train_negative_centerlines + true_path_centerlines,
        "edges_classes": train_negative_classes + true_edges_classes
    }

    # Save the training data to a JSON file
    if debug:
        return
    with open(centerline_path, 'w') as f:
        json.dump(train_data, f, indent=4)


In [11]:
from tqdm import tqdm

#from path_neural_networks.data.compute_samples import process_case

for i in tqdm(range(len(os.listdir(gt_folder)))):
    process_case(i, data_dir, centerlines_folder, max_dist, oversampling_max_dist, n_closest, display=True, debug=True)
    break

  0%|          | 0/353 [00:00<?, ?it/s]

  0%|          | 0/353 [00:06<?, ?it/s]


In [10]:
print(f"Train data centerlines saved to {centerlines_folder}, total {len(os.listdir(centerlines_folder))} centerlines files.")

Train data centerlines saved to /home/morand/afs/EVAPORE/data/PERSEVERE/centerlines/euclidean_lt_100_centerlines, total 353 centerlines files.


Now that we have the centerlines data to train the model on, you can continue on the [training path classification model notebook (5)](./05_train_path_classification_model.ipynb)